In [1]:
!pip install ultralytics streamlit -q

from ultralytics import YOLO
import cv2
import numpy as np
print("✅ Setup Complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 732.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 66.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Setup Complete!


# Day 16: Real-time WebCam Detection + Streamlit Dashboard

## New Concepts Learned Today

**Note 1: Real-time Processing**
- Processing live video from webcam
- Maintaining smooth FPS (Frames Per Second)

**Note 2: Streamlit Dashboard**
- Building professional web interfaces
- Real-time video streaming in browser
- Adding controls (confidence, model selection)

**Note 3: Why this is important?**
- This is what companies actually use in production
- Very strong for hackathons and job portfolios

In [2]:
model = YOLO("yolo11s.pt")

def real_time_detection(confidence=0.25):
    cap = cv2.VideoCapture(0)   # 0 = default webcam

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Run detection
        results = model(frame, conf=confidence, verbose=False)
        annotated = results[0].plot()

        # Show in window (for Colab)
        cv2.imshow("Real-time Detection", annotated)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [3]:
import streamlit as st
from ultralytics import YOLO
import cv2
from PIL import Image
import numpy as np

st.title("🚀 Real-time Smart Detection Dashboard")
st.markdown("### YOLO11 | Day 16 Advanced Project")

# Sidebar controls
confidence = st.slider("Confidence Threshold", 0.1, 0.95, 0.25)
model_choice = st.selectbox("Select Model", ["yolo11s.pt", "yolo11m.pt"])

# Load model
model = YOLO(model_choice)

# Webcam feed
run = st.checkbox("Start Webcam")

if run:
    cap = cv2.VideoCapture(0)
    frame_placeholder = st.empty()

    while run:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=confidence, verbose=False)
        annotated = results[0].plot()

        # Convert to RGB for Streamlit
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        frame_placeholder.image(annotated_rgb, channels="RGB")

    cap.release()

2026-06-13 15:23:09.882 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 15:23:09.969 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-06-13 15:23:09.970 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 15:23:09.970 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 15:23:09.972 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 15:23:09.973 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 15:23:09.974 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-13 15:23:09.975 Thread 'MainThread': mi

### Why Streamlit isn't working directly in the cell

The error `Session state does not function when running a script without streamlit run` occurs because Streamlit applications are designed to be run as standalone processes, not directly within a notebook cell like a regular Python script. When you run a Streamlit script, it starts a web server that you then access through a browser.

To make your Streamlit app accessible from Colab, you need to:

1.  **Save the Streamlit code to a `.py` file.**
2.  **Install `ngrok`** (or `localtunnel`) to create a public URL for your local Streamlit server.
3.  **Run the Streamlit app** in the background and expose it using `ngrok`.

Let's set this up!

In [ ]:
%%writefile app.py

import streamlit as st
from ultralytics import YOLO
import cv2
from PIL import Image
import numpy as np

st.title("🚀 Real-time Smart Detection Dashboard")
st.markdown("### YOLO11 | Day 16 Advanced Project")

# Sidebar controls
confidence = st.slider("Confidence Threshold", 0.1, 0.95, 0.25)
model_choice = st.selectbox("Select Model", ["yolo11s.pt", "yolo11m.pt"]) # Make sure 'yolo11m.pt' is available or downloaded

# Load model
# Check if the model file exists, otherwise download it
if not YOLO(model_choice).exists():
    YOLO(model_choice) # This will download the model if it doesn't exist
model = YOLO(model_choice)

# Webcam feed
run = st.checkbox("Start Webcam")

if run:
    st.warning("Webcam support in Streamlit within Colab can be tricky due to browser security and sandbox limitations. You might need to allow webcam access in your browser.")

    # In a real deployed Streamlit app, you might use a library like `webrtc-streamlit`
    # to handle webcam streams more robustly. For simple testing, cv2.VideoCapture(0)
    # may or may not work depending on your environment/browser settings.

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        st.error("Could not open webcam. Make sure your webcam is available and allowed by the browser.")

    frame_placeholder = st.empty()

    while run and cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            st.warning("Failed to read frame from webcam.")
            break

        # Reduce frame size for faster processing in Colab if needed
        # frame = cv2.resize(frame, (640, 480))

        results = model(frame, conf=confidence, verbose=False)
        annotated = results[0].plot()

        # Convert to RGB for Streamlit
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        frame_placeholder.image(annotated_rgb, channels="RGB", use_column_width=True)

    cap.release()
st.info("Webcam stopped or not started.")


The code above saves your Streamlit application to a file named `app.py`.

Now, let's install `ngrok` and run the Streamlit app through it. This will provide a public URL to access your dashboard.

**Note:** You might need to get an `ngrok` authentication token from [ngrok.com](https://ngrok.com/) for persistent tunnels, though a temporary one usually works for short sessions.

In [ ]:
!pip install pyngrok -q

from pyngrok import ngrok
import os

# Replace 'YOUR_NGROK_AUTH_TOKEN' with your actual ngrok auth token if you have one
# This is optional for temporary tunnels but recommended for stability.
# ngrok_auth_token = "YOUR_NGROK_AUTH_TOKEN"
# if ngrok_auth_token:
#     ngrok.set_auth_token(ngrok_auth_token)

# Kill any running ngrok tunnels (useful if you run this cell multiple times)
!kill -9 $(lsof -t -i:8501) > /dev/null 2>&1 || true

# Start ngrok tunnel for Streamlit (default port 8501)
public_url = ngrok.connect(8501)
print(f"Streamlit App URL: {public_url}")

# Run Streamlit app in the background
# The '&' runs the command in the background, allowing the cell to complete execution
# The output of Streamlit will still appear in the cell output below.
!streamlit run app.py &

After running the cell above, you will see a `Streamlit App URL:` printed. Click on that URL to open your real-time detection dashboard in a new tab.

**Important Considerations for Webcam in Colab/Streamlit:**

*   **Browser Permissions:** You'll likely need to grant your browser permission to access the webcam when prompted by the Streamlit app.
*   **Colab Environment:** Due to the sandboxed nature of Colab and how browsers handle webcam access for web applications, direct `cv2.VideoCapture(0)` might not always work reliably through `ngrok`. For more robust webcam integration in Streamlit, especially for deployment, libraries like `webrtc-streamlit` are often used.